# V2: Korpusi i BPE

**I051 Računalno jezikoslovlje** · vježbe uz 2. predavanje (Jurafsky & Martin, *Speech and Language Processing*, 3. izd., pogl. 2)

Na predavanju: što je riječ, korpusi, Heapsov zakon i BPE. Danas sve to mjerimo na pravim korpusima, u četiri koraka:

| | Predavanje | Vježbe |
|---|---|---|
| 1 | Koliko riječi ima jezik? | korpusi u NLTK-u, pojavnice i različnice |
| 2 | Tekst ne nastaje niotkuda | žanrovi u korpusu Brown |
| 3 | Heapsov zakon | rast rječnika: engleski i hrvatski |
| 4 | BPE | učenje od nule i višejezično usitnjavanje |

## 0. Priprema u VSCode-u

1. **File → Open Folder…** → mapa `vjezbe` (tjedan 2).
2. U terminalu (**Ctrl + `**):
   ```bash
   python -m venv .venv
   .venv\Scripts\activate        # macOS/Linux: source .venv/bin/activate
   pip install -r requirements.txt
   ```
3. U bilježnici gore desno: **Select Kernel → .venv**. Ćeliju pokrećete sa **Shift + Enter**.

**NLTK** (*Natural Language Toolkit*) je Python biblioteka za obradu jezika s desecima korpusa. Koristimo dva engleska:

* **Brown** (1961.) — prvi veliki računalni korpus američkog engleskog: oko milijun riječi iz 500 tekstova u 15 žanrova,
* **Gutenberg** — 18 knjiga iz projekta Gutenberg, među njima tri Shakespeareove drame,

i hrvatski korpus iz V1 (UD Croatian SET): `data/hr_train.txt` i `data/hr_test.txt`, rečenica po retku, tokeni odvojeni razmakom.

Pri prvom pokretanju NLTK preuzima korpuse s interneta. *Ako preuzimanje javi* `Security Violation … proxied fetch` *(mreža s posredničkim poslužiteljem), prije pokretanja VSCode-a postavite varijablu okoline* `NLTK_ALLOW_PROXIED_URLOPEN=1`.

In [ ]:
import math
from collections import Counter
import nltk
import matplotlib.pyplot as plt

nltk.download(["brown", "gutenberg"], quiet=True)     # first run only: downloads the corpora
from nltk.corpus import brown, gutenberg

def read_tokens(path):
    # all tokens of a file with one sentence per line, tokens separated by spaces
    return [w for line in open(path, encoding="utf-8") for w in line.split()]

hr_train = read_tokens("data/hr_train.txt")
hr_test = read_tokens("data/hr_test.txt")
print(brown.words()[:12])
print(hr_train[:12])

## 1. Korpusi u NLTK-u

Korpus u NLTK-u ima metode `fileids()` (datoteke), `words()` (svi tokeni) i, za Brown, `categories()` (žanrovi). `words()` prima i datoteke ili žanrove: `brown.words(categories="romance")`.

In [ ]:
print(brown.categories())
print(gutenberg.fileids())
print(brown.words(categories="romance")[:20])

**Konkordancija** prikazuje svako pojavljivanje riječi s okolnim tekstom. `nltk.Text` omata bilo koju listu tokena — pa i naš hrvatski korpus:

In [ ]:
nltk.Text(hr_train).concordance("jezik", lines=5)

### Zadatak 1

Napišite `stats(tokens)` koja vraća par $(N, |V|)$: $N$ je broj **pojavnica** (svih pojavljivanja riječi), a $|V|$ broj **različnica** (različitih riječi). Brojimo samo riječi od slova, malim slovima — za to služi zadana funkcija `words_of`. Zatim usporedite rezultat s tablicom »Koliko riječi ima jezik?« s predavanja.

In [ ]:
def words_of(tokens):
    # only alphabetic tokens, lowercased: punctuation and numbers are left out
    return [w.lower() for w in tokens if w.isalpha()]

def stats(tokens):
    # TODO: return (N, |V|) for words_of(tokens)
    pass

shakespeare = gutenberg.words(["shakespeare-caesar.txt", "shakespeare-hamlet.txt", "shakespeare-macbeth.txt"])
for name, tokens in [("Brown", brown.words()), ("Shakespeare (3 drame)", shakespeare), ("UD Croatian SET", hr_train)]:
    print(f"{name:22}", stats(tokens))

**Za razmisliti:** hrvatski korpus ima oko 7 puta manje pojavnica od Browna. Usporedite broj različnica — kako to objašnjavate?

## 2. Žanr: tko je, kada i zašto napisao tekst

`ConditionalFreqDist` broji riječi odvojeno za svaki **uvjet** — ovdje žanr; `tabulate` ispisuje tablicu. Klasičan primjer iz knjige NLTK su modalni glagoli:

In [ ]:
genres = ["news", "religion", "hobbies", "science_fiction", "romance", "humor"]
cfd = nltk.ConditionalFreqDist((g, w.lower()) for g in genres for w in brown.words(categories=g))
cfd.tabulate(conditions=genres, samples=["can", "could", "may", "might", "must", "will"])

### Zadatak 2

Napišite `pronoun_share(genre)` koja vraća udio riječi žanra (`words_of`) koje su zamjenice 1. i 2. lica iz skupa `pronouns`. Ispišite žanrove od najvećeg do najmanjeg udjela.

In [ ]:
pronouns = {"i", "me", "my", "you", "your", "we", "us", "our"}

def pronoun_share(genre):
    # TODO: share of words_of(brown.words(categories=genre)) that are in pronouns
    pass

for genre in brown.categories():
    print(f"{genre:16}", pronoun_share(genre))

**Za razmisliti:** zašto se žanrovi toliko razlikuju? Bi li model naučen samo na vijestima dobro razumio razgovor ili roman?

## 3. Heapsov zakon: engleski i hrvatski

S predavanja: $|V| = kN^\beta$, gdje su $k$ i $\beta$ pozitivne konstante, $0 < \beta < 1$. Usporedimo rast rječnika na **jednakom** broju pojavnica. Brown počinje vijestima, pa uspoređujemo vijesti s vijestima.

Ako znamo $|V_1|$ za $N_1$ i $|V_2|$ za $N_2$ pojavnica, onda $\beta = \log(|V_2| / |V_1|) \,/\, \log(N_2 / N_1)$.

In [ ]:
hr_words = words_of(hr_train)
en_words = words_of(brown.words())[:len(hr_words)]          # same N as the Croatian corpus
print(len(hr_words), "Croatian words,", len(en_words), "English words")

### Zadatak 3

Napišite `vocab_growth(words, step)` koja vraća listu parova $(N, |V|)$ nakon svakih `step` riječi.

In [ ]:
def vocab_growth(words, step=5000):
    # TODO: list of (N, |V|) after every `step` words
    pass

In [ ]:
for name, words in [("engleski (Brown)", en_words), ("hrvatski (UD)", hr_words)]:
    points = vocab_growth(words)
    plt.plot([n for n, v in points], [v for n, v in points], label=name)
    (n1, v1), (n2, v2) = points[len(points) // 2 - 1], points[-1]      # N and 2N
    print(f"{name:17} N = {n2}   |V| = {v2}   β ≈ {math.log(v2 / v1) / math.log(n2 / n1):.2f}")

plt.xlabel("N — broj pojavnica")
plt.ylabel("|V| — broj različnica")
plt.legend()
plt.show()

**Za razmisliti:** za engleske korpuse $\beta$ je obično 0,44–0,56. Zašto je za hrvatski veći? Što to znači za model kojemu je rječnik popis riječi?

## 4. BPE implementacija

Algoritam s predavanja (SLP3, sl. 2.6): krenemo od znakova i $k$ puta spojimo najčešći par susjednih simbola. Kao u knjizi:

* spajanja ne prelaze granice riječi — učimo iz liste riječi s brojem pojavljivanja,
* znak `▁` označava početak riječi,
* `vocab` pamti svaku riječ kao n-torku simbola s brojem pojavljivanja, npr. `('▁', 'j', 'e'): 4674`.

In [ ]:
def word_counts(words):
    # every word with the start-of-word symbol ▁ and its count
    return Counter("▁" + w for w in words)

def pair_counts(vocab):
    # how often each pair of adjacent symbols occurs (weighted by word counts)
    pairs = Counter()
    for symbols, c in vocab.items():
        for pair in zip(symbols, symbols[1:]):
            pairs[pair] += c
    return pairs

def merge(symbols, pair):
    # replace every occurrence of pair in symbols with one merged symbol
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == pair:
            out.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            out.append(symbols[i])
            i += 1
    return tuple(out)

print(merge(("▁", "r", "e", "n", "e", "w"), ("n", "e")))

### Zadatak 4

Napišite `learn_bpe(counts, k)` koja vraća listu od `k` spajanja (parova simbola), redom kojim su naučena. U svakom koraku: prebrojite parove (`pair_counts`), uzmite najčešći, zapamtite ga i spojite ga u svim riječima (`merge`).

In [ ]:
def learn_bpe(counts, k):
    vocab = {tuple(w): c for w, c in counts.items()}      # every word split into characters
    merges = []
    # TODO: k times: count pairs, take the most frequent, remember it, merge it in every word
    return merges

Naučimo dva tokenizatora s istim $k = 300$: jedan na hrvatskom, drugi na engleskom tekstu iste duljine.

In [ ]:
merges_hr = learn_bpe(word_counts(hr_words), 300)       # about half a minute
merges_en = learn_bpe(word_counts(en_words), 300)
print("HR:", ["".join(pair) for pair in merges_hr[:12]])
print("EN:", ["".join(pair) for pair in merges_en[:12]])

**Koder** primjenjuje naučena spajanja na novu riječ, **istim redom** kojim su naučena.

In [ ]:
def encode(word, merges):
    symbols = tuple("▁" + word)
    for pair in merges:
        symbols = merge(symbols, pair)
    return symbols

print(f"{'':14} {'HR-BPE':28} EN-BPE")
for w in ["jezikoslovlje", "računalno", "predsjednik", "neovisnosti", "president", "independence"]:
    print(f"{w:14} {' '.join(encode(w, merges_hr)):28} {' '.join(encode(w, merges_en))}")

**Višejezično usitnjavanje.** **Fertilnost** tokenizatora je prosječan broj tokena po riječi (kao u V1). Izmjerimo je na tekstovima koje tokenizatori nisu vidjeli.

In [ ]:
def fertility(words, merges):
    lengths = {w: len(encode(w, merges)) for w in set(words)}      # encode each distinct word once
    return sum(lengths[w] for w in words) / len(words)

hr_test_words = words_of(hr_test)
en_test_words = words_of(brown.words())[-len(hr_test_words):]     # English text not used for training
for name, merges in [("HR-BPE", merges_hr), ("EN-BPE", merges_en)]:
    print(f"{name}   hrvatski test: {fertility(hr_test_words, merges):.2f}   engleski test: {fertility(en_test_words, merges):.2f}")

**Za razmisliti:** usporedite s predavanjem (»Višejezični tokenizatori i hrvatski«). Što usitnjavanje znači za duljinu konteksta i cijenu korištenja LLM-a na hrvatskom? Zašto je hrvatski i s vlastitim tokenizatorom »skuplji« od engleskog? Jesu li dijelovi riječi morfemi?

## Domaća zadaća

1. **Podatkovna kartica.** Pročitajte `print(brown.readme())` i ispunite podatkovnu karticu s predavanja: motivacija, situacija, jezični varijetet, demografija autora, prikupljanje, distribucija. Što nedostaje?
2. **Veličina rječnika.** Ponovite dio 4 za $k$ = 100, 300 i 1000 spajanja. Kako se mijenja fertilnost na hrvatskom testu?
3. **Shakespeare i Austen.** Usporedite $N$, $|V|$ i $\beta$ za `shakespeare-hamlet.txt` i `austen-emma.txt`.